<a name='top' />

# Shadow Hubs in Global Oil Trade
### Network Analytics & Sanctions Overlay

**Team:** Keerti Rawat · Muskan Khepar · Nikhil Kumar · Stiles Clements · Zan Merrill  
**Course:** Social Media & Network Analytics — Spring 2026  

Multi-layer analysis of illicit oil trade networks combining OFAC sanctions data, UN Comtrade bilateral flows (2019–2024), and graph-theoretic "shadow hub" detection.

---

### Table of Contents

**Data Wrangling & Network Construction**

1. <a href=#ofac>OFAC Dataset</a>
2. <a href=#locport>LOCPORT Dataset</a>
3. <a href=#comtrade>UN Comtrade Oil Trade Flows</a>
4. <a href=#diagnostics>Network Diagnostics</a>

**Graph Database & Visualization**

5. <a href=#auradb>Graph Database (Neo4j AuraDB)</a>
6. <a href=#globe>Interactive Globe Visualization</a>

**Next Steps**

7. <a href=#analytics>Network Analytics</a>
8. <a href=#graphrag>GraphRAG Q&A Pipeline</a>


<a name='ofac' />

# OFAC Dataset

<a href=#top>↑ Return to Top of Notebook</a>


In [ ]:
import pandas as pd
import numpy as np
import csv

# =========================
# 1) Load SDN core
# =========================
SDN_COLS = [
    "ent_num", "sdn_name", "sdn_type", "program", "title", "call_sign",
    "vess_type", "tonnage", "grt", "vess_flag", "vess_owner", "remarks"
]

sdn = pd.read_csv(
    "sdn.csv",
    header=None,
    names=SDN_COLS,
    dtype=str,
    na_values=["-0-"],
    keep_default_na=False,
)

sdn["ent_num"] = sdn["ent_num"].astype(str).str.strip()

# =========================
# 2) Load addresses (country signal)
# =========================
ADD_COLS = ["ent_num", "add_num", "address",
            "city_state_postal", "country", "add_remarks"]

add = pd.read_csv(
    "add.csv",
    header=None,
    names=ADD_COLS,
    dtype=str,
    na_values=["-0-"],
    keep_default_na=False,
)

add["ent_num"] = add["ent_num"].astype(str).str.strip()
add["country"] = add["country"].astype(str).str.strip()

add_nonempty = add[(add["country"] != "") & (add["country"].notna())]

country_list = (
    add_nonempty.groupby("ent_num")["country"]
    .apply(lambda s: sorted(set([x for x in s if x and x != "-0-"])))
    .reset_index()
    .rename(columns={"country": "countries"})
)

primary_country = (
    add_nonempty.groupby("ent_num")["country"]
    .agg(lambda s: s.value_counts().index[0])
    .reset_index()
    .rename(columns={"country": "primary_country"})
)

address_count = (
    add.groupby("ent_num")["add_num"]
    .count()
    .reset_index()
    .rename(columns={"add_num": "address_count"})
)

# =========================
# 3) Load spillover remarks (PATCHED)
#    Your file is comma-delimited with quotes, NOT tab-delimited.
# =========================
rows = []
with open("sdn_comments.csv", "r", encoding="utf-8",
          errors="replace", newline="") as f:
    reader = csv.reader(f, delimiter=",", quotechar='"')
    for row in reader:
        if not row:
            continue

        ent = row[0].strip().rstrip(",")
        spill = ",".join(row[1:]).strip()  # safe if row splits into >2 cols
        if ent:
            rows.append((ent, spill))

comments = pd.DataFrame(rows, columns=["ent_num", "remarks_spillover"])

# If an ent_num appears multiple times, concatenate in file order
comments_agg = (
    comments.groupby("ent_num")["remarks_spillover"]
    .apply(lambda s: " ".join([x.strip() for x in s if x and x.strip()]))
    .reset_index()
)

# =========================
# 4) Merge into final entity dataframe
# =========================
final = sdn.merge(comments_agg, on="ent_num", how="left")
final = final.merge(country_list, on="ent_num", how="left")
final = final.merge(primary_country, on="ent_num", how="left")
final = final.merge(address_count, on="ent_num", how="left")

final["remarks"] = final["remarks"].fillna("").astype(str)
final["remarks_spillover"] = final["remarks_spillover"].fillna("").astype(str)

final["remarks_full"] = (
    final["remarks"].str.strip()
    + np.where(
        final["remarks_spillover"].str.strip() != "",
        " " + final["remarks_spillover"].str.strip(),
        ""
    )
).str.strip()

final["has_country"] = final["countries"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

# =========================
# Optional: sanity checks
# =========================
print("Entities:", len(final))
print("Spillover rows:", len(comments))
print(
    "Overlap (SDN ∩ comments):",
    len(set(sdn["ent_num"]) & set(comments["ent_num"]))
)
print(
    "With spillover:",
    (final["remarks_spillover"].str.strip() != "").mean()
)
print("With any country:", final["has_country"].mean())

# final is your “one row per OFAC SDN entity” dataframe

Entities: 18705
Spillover rows: 31
Overlap (SDN ∩ comments): 31
With spillover: 0.0016038492381716118
With any country: 0.8291900561347233


In [ ]:
country_edges = (
    add_nonempty.groupby(["ent_num", "country"])
    .size()
    .reset_index(name="addr_count")
)

<a name='locport' />

# LOCPORT Dataset

<a href=#top>↑ Return to Top of Notebook</a>


In [ ]:
import pandas as pd
import numpy as np
import re

# ----------------------------
# Helpers
# ----------------------------
def norm_unlocode(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x).upper().strip()
    # Remove spaces, slashes, dashes, etc. Keep only A-Z0-9
    s = re.sub(r"[^A-Z0-9]", "", s)
    return s

def parse_unlocode_coord(coord: str):
    """
    UN/LOCODE coordinate format: '4230N 00131E' -> (lat, lon) decimals
    """
    if coord is None or (isinstance(coord, float) and np.isnan(coord)):
        return (np.nan, np.nan)
    s = str(coord).strip().upper()
    if not s:
        return (np.nan, np.nan)

    m = re.match(r"^(\d{2})(\d{2})([NS])\s+(\d{3})(\d{2})([EW])$", s)
    if not m:
        return (np.nan, np.nan)

    lat_deg = int(m.group(1))
    lat_min = int(m.group(2))
    lat_hemi = m.group(3)

    lon_deg = int(m.group(4))
    lon_min = int(m.group(5))
    lon_hemi = m.group(6)

    lat = lat_deg + lat_min / 60.0
    lon = lon_deg + lon_min / 60.0

    if lat_hemi == "S":
        lat = -lat
    if lon_hemi == "W":
        lon = -lon

    return (lat, lon)

def to_float(x):
    try:
        s = str(x).strip()
        if s == "":
            return np.nan
        return float(s)
    except Exception:
        return np.nan

def coalesce(a: pd.Series, b: pd.Series):
    # Prefer a if not empty/null, else b
    a2 = a.copy()
    if a2.dtype == object:
        a2 = a2.replace("", np.nan)
    return a2.combine_first(b)

# ----------------------------
# 1) Load UN/LOCODE code-list
# ----------------------------
UNLOCODE_FILE = "code-list.csv"   # change if needed
unloc = pd.read_csv(UNLOCODE_FILE, dtype=str, keep_default_na=False)

unloc["Country"] = unloc["Country"].astype(str).str.strip().str.upper()
unloc["Location"] = unloc["Location"].astype(str).str.strip().str.upper()
unloc["unlocode"] = (unloc["Country"] + unloc["Location"]).apply(norm_unlocode)

# Parse UN/LOCODE coords (optional fallback)
latlon = unloc["Coordinates"].apply(parse_unlocode_coord)
unloc["lat_unloc"] = latlon.apply(lambda x: x[0])
unloc["lon_unloc"] = latlon.apply(lambda x: x[1])

# Keep only port-type locations: Function includes '1'
unloc_ports = unloc[
    unloc["Function"].astype(str).str.contains("1", na=False)
].copy()

unloc_ports = unloc_ports.rename(
    columns={
        "Country": "country_iso2_unloc",
        "Location": "location_code",
        "Name": "name_unloc",
        "Subdivision": "subdivision",
        "Status": "status",
        "Function": "function",
        "IATA": "iata",
        "Remarks": "remarks_unloc",
    }
)

unloc_ports["has_unlocode"] = True

# ----------------------------
# 2) Load WPI (your schema)
# ----------------------------
WPI_FILE = "UpdatedPub150.csv"    # change if needed
wpi = pd.read_csv(WPI_FILE, dtype=str, keep_default_na=False)

# Use EXACT column names from your printout
wpi["unlocode"] = wpi["UN/LOCODE"].apply(norm_unlocode)
wpi["name_wpi"] = wpi["Main Port Name"].astype(str).str.strip()
wpi["country_iso2_wpi"] = wpi["Country Code"].astype(str).str.strip().str.upper()

wpi["lat_wpi"] = wpi["Latitude"].apply(to_float)
wpi["lon_wpi"] = wpi["Longitude"].apply(to_float)

# Optional: keep a few oil-relevant feature columns for later scoring
keep_wpi_cols = [
    "unlocode",
    "name_wpi",
    "country_iso2_wpi",
    "lat_wpi",
    "lon_wpi",
    "Region Name",
    "Harbor Size",
    "Harbor Type",
    "Harbor Use",
    "Oil Terminal Depth (m)",
    "Facilities – Oil Terminal",
    "Facilities – Liquid Bulk",
    "Supplies – Fuel Oil",
    "Supplies – Diesel Oil",
]

# Some WPI exports use slightly different dash characters; keep only cols that exist
keep_wpi_cols = [c for c in keep_wpi_cols if c in wpi.columns]
wpi_small = wpi[keep_wpi_cols].copy()
wpi_small["has_wpi"] = True

# ----------------------------
# 3) Merge UN/LOCODE ports with WPI
# ----------------------------
ports_master = unloc_ports.merge(wpi_small, on="unlocode", how="left")
ports_master["has_wpi"] = ports_master["has_wpi"].astype("boolean").fillna(False)

# Best fields: prefer WPI, fallback to UN/LOCODE
ports_master["port_name"] = coalesce(
    ports_master.get("name_wpi", pd.Series([np.nan] * len(ports_master))),
    ports_master["name_unloc"],
)

# Force country ISO2 from UN/LOCODE (most reliable)
ports_master["country_iso2"] = ports_master["unlocode"].str[:2]

# (Optional) keep WPI's country field separately for reference
if "country_iso2_wpi" in ports_master.columns:
    ports_master = ports_master.rename(
        columns={"country_iso2_wpi": "wpi_country_raw"}
    )

ports_master["lat"] = coalesce(
    ports_master.get("lat_wpi", pd.Series([np.nan] * len(ports_master))),
    ports_master["lat_unloc"],
)
ports_master["lon"] = coalesce(
    ports_master.get("lon_wpi", pd.Series([np.nan] * len(ports_master))),
    ports_master["lon_unloc"],
)

# ----------------------------
# 4) Final output columns
# ----------------------------
base_cols = [
    "unlocode",
    "country_iso2",
    "location_code",
    "port_name",
    "lat",
    "lon",
    "iata",
    "subdivision",
    "status",
    "function",
    "has_unlocode",
    "has_wpi",
]

feature_cols = [c for c in keep_wpi_cols if c not in {
    "unlocode", "name_wpi", "country_iso2_wpi", "lat_wpi", "lon_wpi"
}]
audit_cols = [
    "name_unloc",
    "lat_unloc",
    "lon_unloc",
    "remarks_unloc",
]
# keep only if present (safe)
audit_cols = [c for c in audit_cols if c in ports_master.columns]

ports_master = ports_master[base_cols + feature_cols + audit_cols].copy()

print("UN/LOCODE port rows:", len(unloc_ports))
print("Ports matched to WPI:", ports_master["has_wpi"].mean())
print(
    "Ports with coords:",
    (ports_master["lat"].notna() & ports_master["lon"].notna()).mean()
)

ports_master.to_csv("ports_master.csv", index=False)
print("Wrote ports_master.csv")

UN/LOCODE port rows: 17616
Ports matched to WPI: 0.16420109958623816
Ports with coords: 0.7438077424474295
Wrote ports_master.csv


/var/folders/0m/sl_d7m1d7tj_3zgcw6mpcc840000gn/T/ipykernel_53046/4071005351.py:140: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ports_master["has_wpi"] = ports_master["has_wpi"].fillna(False)


In [ ]:
print("rows:", len(ports_master))
print("unique unlocode:", ports_master["unlocode"].nunique())
print("match rate (has_wpi):", ports_master["has_wpi"].mean())
print("coords coverage:", (ports_master["lat"].notna() &
                          ports_master["lon"].notna()).mean())

# check if any unlocode appears multiple times
dup = ports_master["unlocode"].value_counts()
print("dups:", (dup > 1).sum())

rows: 17643
unique unlocode: 17540
match rate (has_wpi): 0.16420109958623816
coords coverage: 0.7438077424474295
dups: 103


In [ ]:
pm = ports_master.copy()

pm["has_coords"] = pm["lat"].notna() & pm["lon"].notna()

pm = pm.sort_values(
    by=["unlocode", "has_wpi", "has_coords"],
    ascending=[True, False, False]
)

ports_master_dedup = pm.drop_duplicates(subset=["unlocode"], keep="first").drop(
    columns=["has_coords"]
)

print("rows:", len(ports_master_dedup))
print("unique unlocode:", ports_master_dedup["unlocode"].nunique())
print("dups:",
      (ports_master_dedup["unlocode"].value_counts() > 1).sum())

ports_master_dedup.to_csv("ports_master_dedup.csv", index=False)
print("Wrote ports_master_dedup.csv")

rows: 17540
unique unlocode: 17540
dups: 0
Wrote ports_master_dedup.csv


In [ ]:
ports_master_dedup.head()

,unlocode,country_iso2,location_code,port_name,lat,lon,iata,subdivision,status,function,...,has_wpi,Region Name,Harbor Size,Harbor Type,Harbor Use,Oil Terminal Depth (m),name_unloc,lat_unloc,lon_unloc,remarks_unloc
0,AEABU,AE,ABU,Abu al Bukhoosh,25.483333,53.133333,,,RL,1-------,...,False,NaN,NaN,NaN,NaN,NaN,Abu al Bukhoosh,25.483333,53.133333,
4,AEAJM,AE,AJM,Ajman,25.416667,55.433333,QAJ,,RL,1-3-----,...,True,United Arab Emirates -- 48260,Very Small,Open Roadstead,Cargo,0.0,Ajman,NaN,NaN,
7,AEAJP,AE,AJP,Al Jeer Port,26.033333,56.050000,,RK,RL,1-3-----,...,False,NaN,NaN,NaN,NaN,NaN,Al Jeer Port,26.033333,56.050000,
31,AEAMF,AE,AMF,Musaffah,24.383333,54.483333,,AZ,RL,1-3-----,...,False,NaN,NaN,NaN,NaN,NaN,Musaffah,24.383333,54.483333,Function is 1 and coordinates is 2423N 05429E
2,AEAMU,AE,AMU,Abu Musa,25.866667,55.016667,,,RL,1-------,...,False,NaN,NaN,NaN,NaN,NaN,Abu Musa,25.866667,55.016667,


<a name='comtrade' />

# UN Comtrade oil trade flows

<a href=#top>↑ Return to Top of Notebook</a>


In [ ]:
! pip install comtradeapicall


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import pandas as pd

# pip install comtradeapicall
import comtradeapicall

SUBSCRIPTION_KEY = os.environ["COMTRADE_API_KEY"]   # put your key in env
OUTDIR = "comtrade_oil"
os.makedirs(OUTDIR, exist_ok=True)

YEARS = list(range(2019, 2025))  # 2019-2024; extend if you want
CMD = "2709,2710"                # HS headings (if 2709 errors, switch to 270900)

def fetch_year(year: int) -> pd.DataFrame:
    df = comtradeapicall.getFinalData(
        SUBSCRIPTION_KEY,
        typeCode="C",        # goods
        freqCode="A",        # annual
        clCode="HS",
        period=str(year),
        reporterCode=None,   # all reporters
        partnerCode=None,    # all partners
        partner2Code=None,   # REQUIRED by your version
        customsCode=None,    # REQUIRED by your version
        motCode=None,        # REQUIRED by your version
        cmdCode=CMD,
        flowCode="X",        # exports only
    )
    return df

all_years = []
for y in YEARS:
    print("Pulling year:", y)
    dfy = fetch_year(y)
    dfy.to_csv(f"{OUTDIR}/oil_exports_{y}.csv", index=False)
    print("rows:", len(dfy))
    all_years.append(dfy)

trade = pd.concat(all_years, ignore_index=True)
trade.to_csv(f"{OUTDIR}/oil_exports_2019plus.csv", index=False)
print("Wrote:", f"{OUTDIR}/oil_exports_2019plus.csv", "rows:", len(trade))

Pulling year: 2019
rows: 34018
Pulling year: 2020
rows: 34316
Pulling year: 2021
rows: 35285
Pulling year: 2022
rows: 38172
Pulling year: 2023
rows: 39165
Pulling year: 2024
rows: 36768
Wrote: comtrade_oil/oil_exports_2019plus.csv rows: 217724


In [ ]:
trade.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20190101,2019,52,2019,112,NaN,NaN,X,...,6.818591e+07,False,0.0,False,NaN,4.611330e+07,4.611330e+07,0,False,True
1,C,A,20190101,2019,52,2019,112,NaN,NaN,X,...,3.301489e+09,False,0.0,False,NaN,2.008522e+09,2.008522e+09,0,False,True
2,C,A,20190101,2019,52,2019,112,NaN,NaN,X,...,1.649400e+09,False,0.0,False,NaN,7.267670e+08,7.267670e+08,0,False,True
3,C,A,20190101,2019,52,2019,112,NaN,NaN,X,...,6.915700e+04,False,0.0,False,NaN,9.090000e+04,9.090000e+04,0,False,True
4,C,A,20190101,2019,52,2019,112,NaN,NaN,X,...,8.175700e+07,False,0.0,False,NaN,4.788940e+07,4.788940e+07,0,False,True


<a name='diagnostics' />

# Network Diagnostics

<a href=#top>↑ Return to Top of Notebook</a>


In [ ]:
import pandas as pd

trade = pd.read_csv("comtrade_oil/oil_exports_2019plus.csv", dtype=str)
print(trade.columns.tolist())
trade.head(2)

['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate']


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20190101,2019,52,2019,112,NaN,NaN,X,...,68185911.0,False,0.0,False,NaN,46113300.0,46113300.0,0,False,True
1,C,A,20190101,2019,52,2019,112,NaN,NaN,X,...,3301489000.0,False,0.0,False,NaN,2008522200.0,2008522200.0,0,False,True


In [ ]:
import pandas as pd
import numpy as np

trade = pd.read_csv(
    "comtrade_oil/oil_exports_2019plus.csv",
    low_memory=False
)

df = trade[trade["flowCode"].astype(str).str.upper().eq("X")].copy()

df["year"] = pd.to_numeric(df["refYear"], errors="coerce").astype("Int64")
df["exporter_code"] = pd.to_numeric(df["reporterCode"], errors="coerce").astype("Int64")
df["importer_code"] = pd.to_numeric(df["partnerCode"], errors="coerce").astype("Int64")
df["trade_value_usd"] = pd.to_numeric(df["primaryValue"], errors="coerce")

df = df.dropna(subset=["year", "exporter_code", "importer_code", "trade_value_usd"])
df = df[df["trade_value_usd"] > 0].copy()

# drop World/invalid codes (0 is commonly World/unknown)
df = df[(df["exporter_code"] > 0) & (df["importer_code"] > 0)].copy()

# optional: drop self-loops
df = df[df["exporter_code"] != df["importer_code"]].copy()

# total oil by edge per year
edge_total = (
    df.groupby(["year", "exporter_code", "importer_code"], as_index=False)
      .agg(trade_value_usd=("trade_value_usd", "sum"))
)

edge_total.to_csv(
    "comtrade_oil/country_edges_oil_total_2019plus.csv",
    index=False
)

print("edge_total rows:", len(edge_total))
print("years:", sorted(edge_total["year"].unique()))
print(edge_total.head(3))

edge_total rows: 47080
years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   year  exporter_code  importer_code  trade_value_usd
0  2019              8            688     1.311290e+08
1  2019             20            724     1.075319e+05
2  2019             24             56     8.942041e+07


In [ ]:
nodes_rep = trade[["reporterCode", "reporterDesc"]].copy()
nodes_rep.columns = ["code", "name"]

nodes_par = trade[["partnerCode", "partnerDesc"]].copy()
nodes_par.columns = ["code", "name"]

nodes = pd.concat([nodes_rep, nodes_par], ignore_index=True)

nodes["code"] = pd.to_numeric(nodes["code"], errors="coerce").astype("Int64")
nodes["name"] = nodes["name"].astype("string").str.strip()

nodes = nodes.dropna(subset=["code", "name"])
nodes = nodes[nodes["code"] > 0].copy()

# keep the most frequent name for a code (stable)import pandas as pd
import numpy as np

TRADE_PATH = "comtrade_oil/oil_exports_2019plus.csv"
OUT_EDGES  = "comtrade_oil/country_edges_oil_2019plus.csv"
OUT_NODES  = "comtrade_oil/country_nodes.csv"

# -----------------------------
# 1) Load trade (your pulled file)
# -----------------------------
trade = pd.read_csv(TRADE_PATH, dtype=str, low_memory=False)

# minimal fields we need
needed = ["refYear", "reporterCode", "partnerCode", "primaryValue"]
missing = [c for c in needed if c not in trade.columns]
if missing:
    raise ValueError(f"Missing expected columns in trade file: {missing}")

trade["year"] = pd.to_numeric(trade["refYear"], errors="coerce").astype("Int64")
trade["exporter_code"] = pd.to_numeric(trade["reporterCode"], errors="coerce").astype("Int64")
trade["importer_code"] = pd.to_numeric(trade["partnerCode"], errors="coerce").astype("Int64")
trade["trade_value_usd"] = pd.to_numeric(trade["primaryValue"], errors="coerce")

trade = trade.dropna(subset=["year", "exporter_code", "importer_code", "trade_value_usd"])
trade = trade[(trade["exporter_code"] > 0) & (trade["importer_code"] > 0)]
trade = trade[trade["trade_value_usd"] > 0]

# Aggregate to country-country edges per year
edges = (
    trade.groupby(["year", "exporter_code", "importer_code"], as_index=False)["trade_value_usd"]
    .sum()
)

print("edges rows:", len(edges))
print("years:", sorted(edges["year"].unique().tolist())[:20])

# -----------------------------
# 2) Pull official reference mappings (code -> name, iso2, iso3)
# -----------------------------
REPORTERS_URL = "https://comtradeapi.un.org/files/v1/app/reference/Reporters.json"
PARTNERS_URL  = "https://comtradeapi.un.org/files/v1/app/reference/partnerAreas.json"

rep_raw = pd.read_json(REPORTERS_URL)
par_raw = pd.read_json(PARTNERS_URL)

rep = pd.json_normalize(rep_raw["results"])
par = pd.json_normalize(par_raw["results"])

rep_ref = rep.rename(columns={
    "reporterCode": "code",
    "reporterDesc": "name",
    "reporterCodeIsoAlpha2": "iso2",
    "reporterCodeIsoAlpha3": "iso3",
    "isGroup": "is_group",
})[["code", "name", "iso2", "iso3", "is_group"]]

par_ref = par.rename(columns={
    "PartnerCode": "code",
    "PartnerDesc": "name",
    "PartnerCodeIsoAlpha2": "iso2",
    "PartnerCodeIsoAlpha3": "iso3",
    "isGroup": "is_group",
})[["code", "name", "iso2", "iso3", "is_group"]]

ref = pd.concat([rep_ref, par_ref], ignore_index=True)

# Make numeric + de-dupe by code (prefer a row that has iso3)
ref["code"] = pd.to_numeric(ref["code"], errors="coerce").astype("Int64")
ref = ref.dropna(subset=["code"])

ref["has_iso3"] = ref["iso3"].notna() & (ref["iso3"].astype(str).str.strip() != "")
ref = (
    ref.sort_values(["code", "has_iso3"], ascending=[True, False])
       .drop_duplicates(subset=["code"], keep="first")
       .drop(columns=["has_iso3"])
)

# -----------------------------
# 3) Build nodes only for codes that appear in edges
# -----------------------------
codes_used = pd.Index(
    pd.unique(edges[["exporter_code", "importer_code"]].to_numpy().ravel())
).dropna()

nodes = ref[ref["code"].isin(codes_used)].copy()
nodes["name"] = nodes["name"].astype("string").fillna(nodes["iso3"]).fillna(nodes["code"].astype(str))

# Optional: flag “special/group” areas (EU, ASEAN, World, etc.)
# Keep them for now; you can filter later depending on your analysis.
nodes.to_csv(OUT_NODES, index=False)
print("nodes:", len(nodes), "w/iso3:", (nodes["iso3"].notna()).mean())

# -----------------------------
# 4) Attach ISO3 to edges (for nicer graphs/plots)
# -----------------------------
ref_iso3 = ref[["code", "iso3"]].copy()

edges = edges.merge(
    ref_iso3.rename(columns={"code": "exporter_code", "iso3": "exporter_iso3"}),
    on="exporter_code",
    how="left",
)
edges = edges.merge(
    ref_iso3.rename(columns={"code": "importer_code", "iso3": "importer_iso3"}),
    on="importer_code",
    how="left",
)

print("edge iso3 coverage:",
      (edges["exporter_iso3"].notna() & edges["importer_iso3"].notna()).mean())

edges.to_csv(OUT_EDGES, index=False)
print("Wrote:", OUT_NODES)
print("Wrote:", OUT_EDGES)
nodes = (
    nodes.groupby("code")["name"]
         .agg(lambda s: s.value_counts().index[0])
         .reset_index()
)

nodes.to_csv("comtrade_oil/country_nodes.csv", index=False)
print("nodes:", len(nodes))
print(nodes.head(5))

edges rows: 47080
years: [2019, 2020, 2021, 2022, 2023, 2024]
nodes: 243 w/iso3: 1.0
edge iso3 coverage: 1.0
Wrote: comtrade_oil/country_nodes.csv
Wrote: comtrade_oil/country_edges_oil_2019plus.csv
nodes: 243
   code            name
0     4     Afghanistan
1     8         Albania
2    10      Antarctica
3    12         Algeria
4    16  American Samoa


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

edges = pd.read_csv("comtrade_oil/country_edges_oil_2019plus.csv")
nodes = pd.read_csv("comtrade_oil/country_nodes.csv")

# Build a name lookup for ISO3 (from edges mapping created earlier)
# If your nodes file has iso3 column, use that instead.
# If not, we’ll infer using the edges file itself.
if "iso3" in nodes.columns:
    iso3_to_name = nodes.dropna(subset=["iso3"]).set_index("iso3")["name"].to_dict()
else:
    # fallback: create iso3->name by taking the most common name per iso3 from edge file merges
    iso3_to_name = {}

# Collapse to total oil per exporter-importer-year (no HS split)
edge_total = (
    edges.groupby(["year", "exporter_iso3", "importer_iso3"], as_index=False)
         .agg(trade_value_usd=("trade_value_usd", "sum"))
)

def shadow_hubs_for_year(df_y: pd.DataFrame) -> pd.DataFrame:
    G = nx.DiGraph()

    for _, r in df_y.iterrows():
        v = float(r["trade_value_usd"])
        if v <= 0:
            continue
        # capacity -> distance (bigger trade = shorter distance)
        dist = 1.0 / np.log1p(v)
        G.add_edge(r["exporter_iso3"], r["importer_iso3"], value=v, dist=dist)

    btw = nx.betweenness_centrality(G, weight="dist", normalized=True)

    vol_out, vol_in = {}, {}
    for u, v, data in G.edges(data=True):
        vol_out[u] = vol_out.get(u, 0.0) + data["value"]
        vol_in[v] = vol_in.get(v, 0.0) + data["value"]

    nodes_list = list(G.nodes())
    res = pd.DataFrame({
        "iso3": nodes_list,
        "betweenness": [btw.get(n, 0.0) for n in nodes_list],
        "trade_out_usd": [vol_out.get(n, 0.0) for n in nodes_list],
        "trade_in_usd": [vol_in.get(n, 0.0) for n in nodes_list],
    })
    res["trade_total_usd"] = res["trade_out_usd"] + res["trade_in_usd"]

    res["log_trade_total"] = np.log1p(res["trade_total_usd"])
    res["btw_z"] = (res["betweenness"] - res["betweenness"].mean()) / (
        res["betweenness"].std(ddof=0) + 1e-12
    )
    res["vol_z"] = (res["log_trade_total"] - res["log_trade_total"].mean()) / (
        res["log_trade_total"].std(ddof=0) + 1e-12
    )
    res["shadow_score"] = res["btw_z"] - res["vol_z"]

    # optional label
    res["name"] = res["iso3"].map(iso3_to_name)

    return res.sort_values("shadow_score", ascending=False)

all_years = []
for y in sorted(edge_total["year"].unique()):
    dfy = edge_total[edge_total["year"] == y].copy()
    hubs = shadow_hubs_for_year(dfy)
    hubs["year"] = y
    hubs.to_csv(f"comtrade_oil/shadow_hubs_{y}.csv", index=False)
    print("year", y, "nodes", len(hubs), "edges", len(dfy))
    all_years.append(hubs)

shadow_all = pd.concat(all_years, ignore_index=True)
shadow_all.to_csv("comtrade_oil/shadow_hubs_2019plus.csv", index=False)
print("Wrote comtrade_oil/shadow_hubs_2019plus.csv rows:", len(shadow_all))

# show top 15 for the most recent year
latest = shadow_all[shadow_all["year"] == shadow_all["year"].max()]
print(latest[["iso3","name","shadow_score","betweenness","trade_total_usd"]].head(15))

year 2019 nodes 237 edges 8111
year 2020 nodes 237 edges 7861
year 2021 nodes 240 edges 7904
year 2022 nodes 241 edges 7907
year 2023 nodes 241 edges 7973
year 2024 nodes 242 edges 7324
Wrote comtrade_oil/shadow_hubs_2019plus.csv rows: 1438
     iso3  name  shadow_score  betweenness  trade_total_usd
1196  USA   NaN      9.327094     0.098202     8.261051e+11
1197  ESH   NaN      3.699147     0.000000     2.026460e+03
1198  CHN   NaN      2.996122     0.040093     1.514957e+11
1199  SGS   NaN      2.844472     0.000000     3.533677e+04
1200  UMI   NaN      2.765161     0.000000     4.606980e+04
1201  TUR   NaN      2.370376     0.034302     1.256925e+11
1202  SGP   NaN      2.313714     0.034284     1.508858e+11
1203  ZAF   NaN      2.198262     0.029236     3.044343e+10
1204  CCK   NaN      2.163439     0.000000     3.446150e+05
1205  A79   NaN      2.160341     0.000000     3.482040e+05
1206  O19   NaN      2.108876     0.000000     4.135990e+05
1207  HMD   NaN      1.893435     0.000

In [ ]:
import pandas as pd

nodes = pd.read_csv("comtrade_oil/country_nodes.csv")
print(nodes.columns.tolist())
nodes.head()

['code', 'name']


,code,name
0,4,Afghanistan
1,8,Albania
2,10,Antarctica
3,12,Algeria
4,16,American Samoa


In [ ]:
import pandas as pd

REPORTERS_URL = "https://comtradeapi.un.org/files/v1/app/reference/Reporters.json"
PARTNERS_URL  = "https://comtradeapi.un.org/files/v1/app/reference/partnerAreas.json"

rep_raw = pd.read_json(REPORTERS_URL)
par_raw = pd.read_json(PARTNERS_URL)

rep = pd.json_normalize(rep_raw["results"])
par = pd.json_normalize(par_raw["results"])

rep_ref = rep.rename(columns={
    "reporterCode": "code",
    "reporterDesc": "name",
    "reporterCodeIsoAlpha2": "iso2",
    "reporterCodeIsoAlpha3": "iso3",
    "isGroup": "is_group",
})[["code","name","iso2","iso3","is_group"]]

par_ref = par.rename(columns={
    "PartnerCode": "code",
    "PartnerDesc": "name",
    "PartnerCodeIsoAlpha2": "iso2",
    "PartnerCodeIsoAlpha3": "iso3",
    "isGroup": "is_group",
})[["code","name","iso2","iso3","is_group"]]

ref = pd.concat([rep_ref, par_ref], ignore_index=True)
ref["code"] = pd.to_numeric(ref["code"], errors="coerce").astype("Int64")
ref = ref.dropna(subset=["code"])

# Keep codes used in your edges file
edges = pd.read_csv("comtrade_oil/country_edges_oil_2019plus.csv")
codes_used = pd.Index(
    pd.unique(
        pd.concat([edges["exporter_code"], edges["importer_code"]], ignore_index=True)
    )
)

nodes = ref[ref["code"].isin(codes_used)].copy()
nodes.to_csv("comtrade_oil/country_nodes.csv", index=False)
print("rewrote nodes:", len(nodes), "cols:", nodes.columns.tolist())

rewrote nodes: 462 cols: ['code', 'name', 'iso2', 'iso3', 'is_group']


In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx

# =========================================================
# Shadow Oil Network (Country-level) — 2019+ (Comtrade)
# Shadow hub score = regression residual:
#   log1p(betweenness) - expected_log1p(betweenness | log1p(volume))
# =========================================================

# --------- CONFIG ----------
EDGES_IN = "comtrade_oil/country_edges_oil_2019plus.csv"      # produced earlier
OUTDIR = "comtrade_oil"
os.makedirs(OUTDIR, exist_ok=True)

YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

# Pruning: keep top (1 - PRUNE_Q) fraction by value per year
PRUNE_Q = 0.00   # 0.60 / 0.70 / 0.80 to tune

# Hub hygiene filters (tune based on how strict you want)
MIN_TRADE_TOTAL_USD = 1e9
MIN_IN_DEG = 10
MIN_OUT_DEG = 10

# Betweenness distance definition
# Using 1/value after pruning works well
def edge_dist(v: float) -> float:
    return 1.0 / v


# --------- LOAD + AGGREGATE YEARLY ISO3 EDGES ----------
edges_raw = pd.read_csv(EDGES_IN)

# Expect columns exporter_iso3/importer_iso3 or exporter_iso/importer_iso
# Normalize names
rename_map = {}
if "exporter_iso" in edges_raw.columns:
    rename_map["exporter_iso"] = "exporter_iso3"
if "importer_iso" in edges_raw.columns:
    rename_map["importer_iso"] = "importer_iso3"
edges_raw = edges_raw.rename(columns=rename_map)

need_cols = {"year", "exporter_iso3", "importer_iso3", "trade_value_usd"}
missing = need_cols - set(edges_raw.columns)
if missing:
    raise ValueError(f"Missing required columns in {EDGES_IN}: {missing}")

# Ensure types
edges_raw["year"] = pd.to_numeric(edges_raw["year"], errors="coerce").astype("Int64")
edges_raw["exporter_iso3"] = edges_raw["exporter_iso3"].astype(str).str.strip()
edges_raw["importer_iso3"] = edges_raw["importer_iso3"].astype(str).str.strip()
edges_raw["trade_value_usd"] = pd.to_numeric(edges_raw["trade_value_usd"], errors="coerce")

edges_raw = edges_raw.dropna(subset=["year", "exporter_iso3", "importer_iso3", "trade_value_usd"])
edges_raw = edges_raw[edges_raw["trade_value_usd"] > 0].copy()

# Aggregate to one edge per (year, exporter_iso3, importer_iso3)
edges_yearly = (
    edges_raw.groupby(["year", "exporter_iso3", "importer_iso3"], as_index=False)
             .agg(trade_value_usd=("trade_value_usd", "sum"))
)

# --------- HELPERS ----------
def build_year_graph(dfy: pd.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()
    for _, r in dfy.iterrows():
        u = r["exporter_iso3"]
        v = r["importer_iso3"]
        val = float(r["trade_value_usd"])
        G.add_edge(u, v, value=val, dist=edge_dist(val))
    return G

def compute_year_metrics(year: int, dfy: pd.DataFrame) -> pd.DataFrame:
    """
    Returns per-node metrics for a year:
    iso3, betweenness, trade_total_usd, in_deg, out_deg, year
    """
    if dfy.empty:
        return pd.DataFrame(columns=[
            "year","iso3","betweenness","trade_total_usd","in_deg","out_deg"
        ])

    # prune micro edges for the year
    thresh = dfy["trade_value_usd"].quantile(PRUNE_Q)
    dfy = dfy[dfy["trade_value_usd"] >= thresh].copy()

    G = build_year_graph(dfy)

    btw = nx.betweenness_centrality(G, weight="dist", normalized=True)

    rows = []
    for n in G.nodes():
        trade_out = sum(G[n][vv]["value"] for vv in G.successors(n))
        trade_in = sum(G[uu][n]["value"] for uu in G.predecessors(n))
        rows.append({
            "year": int(year),
            "iso3": n,
            "betweenness": float(btw.get(n, 0.0)),
            "trade_total_usd": float(trade_in + trade_out),
            "in_deg": int(G.in_degree(n)),
            "out_deg": int(G.out_degree(n)),
            "prune_threshold_usd": float(thresh),
            "edges_after_pruning": int(G.number_of_edges()),
            "nodes_after_pruning": int(G.number_of_nodes()),
        })

    res = pd.DataFrame(rows)

    # ISO3 hygiene (drop aggregates like X2/XX)
    res = res[res["iso3"].astype(str).str.match(r"^[A-Z]{3}$", na=False)].copy()

    return res

def add_shadow_residual(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds regression residual score per year:
      log_btw ~ log_vol
      shadow_resid = log_btw - expected(log_btw|log_vol)
    Also adds:
      log_btw, log_vol, btw_expected
    """
    out = df.copy()
    out["log_btw"] = np.log1p(out["betweenness"])
    out["log_vol"] = np.log1p(out["trade_total_usd"])

    # Fit separately per year (important)
    shadow_resid = []
    btw_expected = []

    for y, g in out.groupby("year", sort=True):
        x = g["log_vol"].to_numpy()
        yv = g["log_btw"].to_numpy()

        # guard against degenerate cases
        if len(g) < 5 or np.allclose(x.var(), 0) or np.allclose(yv.var(), 0):
            pred = np.full(len(g), np.nan)
            resid = np.full(len(g), np.nan)
        else:
            b1, b0 = np.polyfit(x, yv, 1)  # yv ≈ b1*x + b0
            pred = b1 * x + b0
            resid = yv - pred

        btw_expected.extend(pred.tolist())
        shadow_resid.extend(resid.tolist())

    out["btw_expected"] = btw_expected
    out["shadow_resid"] = shadow_resid
    return out

# --------- MAIN: compute per-year nodes, then score ----------
all_years = []
for y in YEARS:
    dfy = edges_yearly[edges_yearly["year"] == y].copy()
    metrics = compute_year_metrics(y, dfy)
    print(
        f"year {y} raw_edges={len(dfy)} "
        f"nodes={metrics['nodes_after_pruning'].max() if len(metrics) else 0} "
        f"edges={metrics['edges_after_pruning'].max() if len(metrics) else 0} "
        f"threshold={metrics['prune_threshold_usd'].max() if len(metrics) else np.nan}"
    )
    all_years.append(metrics)

nodes_all = pd.concat(all_years, ignore_index=True)

# Candidate filters for "middleman hubs"
nodes_all = nodes_all[
    (nodes_all["betweenness"] > 0) &
    (nodes_all["in_deg"] > 0) & (nodes_all["out_deg"] > 0) &
    (nodes_all["trade_total_usd"] >= MIN_TRADE_TOTAL_USD) &
    (nodes_all["in_deg"] >= MIN_IN_DEG) &
    (nodes_all["out_deg"] >= MIN_OUT_DEG)
].copy()

print("candidates after filters:", len(nodes_all))

# Add residual score
scored = add_shadow_residual(nodes_all)

# Rank per year (higher residual = more "shadow hub" for its volume)
scored["shadow_rank"] = scored.groupby("year")["shadow_resid"].rank(
    ascending=False, method="dense"
)

# Write outputs
scored.to_csv(f"{OUTDIR}/shadow_hubs_residual_2019plus.csv", index=False)
print(f"Wrote {OUTDIR}/shadow_hubs_residual_2019plus.csv rows:", len(scored))

# Quick view: top 25 for latest year
latest_year = int(scored["year"].max()) if len(scored) else None
if latest_year is not None:
    top_latest = (
        scored[scored["year"] == latest_year]
        .sort_values(["shadow_resid"], ascending=False)
        .head(25)
        [["iso3","shadow_rank","shadow_resid","betweenness","trade_total_usd","in_deg","out_deg"]]
    )
    print("\nTop shadow hubs (latest year):", latest_year)
    print(top_latest.to_string(index=False))
else:
    print("No scored rows produced — relax filters or check inputs.")

year 2019 raw_edges=8111 nodes=237 edges=8111 threshold=0.001
year 2020 raw_edges=7861 nodes=237 edges=7861 threshold=0.16
year 2021 raw_edges=7904 nodes=240 edges=7904 threshold=0.334
year 2022 raw_edges=7907 nodes=241 edges=7907 threshold=0.09
year 2023 raw_edges=7973 nodes=241 edges=7973 threshold=0.032
year 2024 raw_edges=7324 nodes=242 edges=7324 threshold=0.441
candidates after filters: 413
Wrote comtrade_oil/shadow_hubs_residual_2019plus.csv rows: 413

Top shadow hubs (latest year): 2024
iso3  shadow_rank  shadow_resid  betweenness  trade_total_usd  in_deg  out_deg
 USA          1.0      0.224350     0.349533     8.261051e+11      97      174
 ESP          2.0      0.067697     0.127299     1.909029e+11      76      149
 NLD          3.0      0.057659     0.123997     2.985135e+11      82      177
 SGP          4.0      0.054936     0.108852     1.508858e+11      75      159
 FJI          5.0      0.037603     0.006535     1.015510e+09      21       18
 BOL          6.0      0.0

In [ ]:
import pandas as pd
import numpy as np

shadow_all = pd.read_csv("comtrade_oil/shadow_hubs_residual_2019plus.csv")
nodes = pd.read_csv("comtrade_oil/country_nodes.csv")

# Build ISO3 -> name map (your nodes file should have iso3)
# If your nodes file instead uses column names differently, print(nodes.columns) and adjust.
iso3_to_name = nodes.dropna(subset=["iso3"]).set_index("iso3")["name"].to_dict()
shadow_all["name"] = shadow_all["iso3"].map(iso3_to_name)

# Keep only standard ISO3 alpha codes (drops "A79", "019", etc)
shadow_all = shadow_all[shadow_all["iso3"].astype(str).str.match(r"^[A-Z]{3}$", na=False)].copy()

# Enforce "bridge-ness": betweenness must be > 0
shadow_all = shadow_all[shadow_all["betweenness"] > 0].copy()

# Better scoring: rank betweenness high AND volume low (per year)
def add_rank_score(df):
    df = df.copy()
    df["btw_rank"] = df["betweenness"].rank(pct=True, ascending=True)   # 0..1
    df["vol_rank"] = df["trade_total_usd"].rank(pct=True, ascending=True)
    df["shadow_score_rank"] = df["btw_rank"] - df["vol_rank"]           # high btw, low vol
    return df

shadow_all = shadow_all.groupby("year", group_keys=False).apply(add_rank_score)

shadow_all.to_csv("comtrade_oil/shadow_hubs_2019plus_clean.csv", index=False)
print("Wrote comtrade_oil/shadow_hubs_2019plus_clean.csv rows:", len(shadow_all))

latest = shadow_all[shadow_all["year"] == shadow_all["year"].max()]
print(
    latest.sort_values("shadow_score_rank", ascending=False)
          [["iso3","name","shadow_score_rank","betweenness","trade_total_usd"]]
          .head(25)
)

Wrote comtrade_oil/shadow_hubs_2019plus_clean.csv rows: 413
    iso3                              name  shadow_score_rank  betweenness  \
408  TTO               Trinidad and Tobago           0.575758     0.015163   
371  BOL  Bolivia (Plurinational State of)           0.530303     0.008281   
382  FJI                              Fiji           0.492424     0.006535   
368  SRB                            Serbia           0.439394     0.008368   
378  CIV                     Côte d'Ivoire           0.424242     0.012915   
388  NZL                       New Zealand           0.409091     0.008454   
380  EGY                             Egypt           0.378788     0.013987   
397  HUN                           Hungary           0.378788     0.023064   
390  SEN                           Senegal           0.348485     0.008230   
406  SVN                          Slovenia           0.348485     0.027611   
398  KEN                             Kenya           0.318182     0.004357   
375 

/var/folders/0m/sl_d7m1d7tj_3zgcw6mpcc840000gn/T/ipykernel_53046/1767345314.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  shadow_all = shadow_all.groupby("year", group_keys=False).apply(add_rank_score)


In [ ]:
! pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 32.1 MB/s  0:00:006.0 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import re
import pandas as pd
import numpy as np

# ---- 1) normalize weird placeholder tokens like –0– / -0- / blanks to NA ----
def clean_placeholder(x: str) -> str:
    if x is None:
        return ""
    x = str(x).strip()

    # normalize unicode dashes to regular hyphen
    x = re.sub(r"[–—−]", "-", x)

    # common placeholders
    if x in {"", "-0-", "0", "NA", "N/A", "NONE", "NULL"}:
        return ""
    return x

# ---- 2) normalize region-y strings ----
def normalize_country_text(x: str) -> str:
    x = clean_placeholder(x)
    if not x:
        return ""

    # Strip "Region:" prefixes
    x = re.sub(r"^\s*Region:\s*", "", x, flags=re.IGNORECASE).strip()

    # Fix common punctuation/spacing
    x = re.sub(r"\s+", " ", x).strip()
    return x

# ---- 3) Manual overrides for common OFAC variants -> ISO3 ----
OVERRIDE_TO_ISO3 = {
    "Turkey": "TUR",
    "Korea, North": "PRK",
    "Korea, South": "KOR",
    "Burma": "MMR",
    "Cote d Ivoire": "CIV",
    "North Macedonia, The Republic of": "MKD",
    "Congo, Democratic Republic of the": "COD",
    "Congo, Republic of the": "COG",
    "Bahamas, The": "BHS",
    "Man, Isle of": "IMN",

    # Palestinian Territories
    "Palestinian": "PSE",
    "West Bank": "PSE",
    "Gaza": "PSE",
    "Northern Gaza": "PSE",

    # Ambiguous / contested / not-a-country labels:
    "undetermined": "",

    # If you want a consistent rule, pick ONE:
    # Crimea is commonly treated as Ukraine (UKR) in many datasets.
    "Crimea": "UKR",

    # "Region: Russia" usually means Russia
    "Russia": "RUS",

    # Mali subregion -> Mali
    "Northern Mali": "MLI",

    # Netherlands Antilles is obsolete; choose a rule (often treat as missing)
    "Netherlands Antilles": "",
}

# ---- 4) Apply mapping onto your entity-level OFAC dataframe (final) ----
# expects: final has column "primary_country"
final = final.copy()

final["primary_country_clean"] = final["primary_country"].map(normalize_country_text)

final["primary_country_iso3"] = final["primary_country_clean"].map(
    lambda s: OVERRIDE_TO_ISO3.get(s, np.nan)
)

# Anything not overridden stays NaN for now (you can optionally try fuzzy matching later)
# If you DO want fuzzy matching via pycountry, tell me and I’ll add it cleanly.

# ---- 5) Diagnostics: what still isn't mapped? ----
unmapped = (
    final.loc[
        final["primary_country_clean"].ne("") &
        final["primary_country_iso3"].isna(),
        "primary_country_clean"
    ]
    .value_counts()
    .head(30)
)

print("Top still-unmapped (after overrides):")
print(unmapped)

print("\nCoverage (non-empty primary_country_clean -> mapped iso3):",
      (final["primary_country_clean"].ne("") & final["primary_country_iso3"].notna()).mean()
)

Top still-unmapped (after overrides):
primary_country_clean
Iran                    1483
Mexico                  1070
China                   1010
United Arab Emirates     584
Colombia                 367
Lebanon                  303
Syria                    270
Belarus                  256
Venezuela                235
Ukraine                  219
United Kingdom           160
Pakistan                 142
Yemen                    130
Panama                   129
Iraq                     128
India                    122
Cambodia                 121
Cyprus                   118
Germany                  108
Marshall Islands         106
Singapore                 93
Bulgaria                  89
Italy                     72
Nicaragua                 66
Somalia                   62
Afghanistan               62
Switzerland               62
Guatemala                 55
United States             55
Hong Kong                 54
Name: count, dtype: int64

Coverage (non-empty primary_country_clean -

In [ ]:
# if needed
# pip install pycountry

import re
import pandas as pd
import pycountry


def normalize_country_text(x) -> str:
    if x is None:
        return ""

    x = str(x).strip()

    # normalize unicode dashes
    x = re.sub(r"[–—−]", "-", x)

    # strip leading "Region:" (case-insensitive)
    x = re.sub(r"^\s*Region:\s*", "", x, flags=re.I).strip()

    # collapse whitespace
    x = re.sub(r"\s+", " ", x).strip()

    # placeholders (case-insensitive)
    xl = x.casefold()
    if xl in {"", "-0-", "0", "na", "n/a", "none", "null", "nan"}:
        return ""

    return x


# --- overrides (updated) ---
OVERRIDE_TO_ISO3 = {
    # your existing overrides
    "Turkey": "TUR",
    "Korea, North": "PRK",
    "Korea, South": "KOR",
    "Burma": "MMR",
    "Cote d Ivoire": "CIV",
    "North Macedonia, The Republic of": "MKD",
    "Congo, Democratic Republic of the": "COD",
    "Congo, Republic of the": "COG",
    "Bahamas, The": "BHS",
    "Man, Isle of": "IMN",
    "Palestinian": "PSE",
    "West Bank": "PSE",
    "Gaza": "PSE",
    "Northern Gaza": "PSE",
    "undetermined": None,
    "Crimea": "UKR",
    "Russia": "RUS",
    "Northern Mali": "MLI",
    "Netherlands Antilles": None,
    "Hong Kong": "HKG",

    # recommended additions
    "Kosovo": "XKX",
    "The Gambia": "GMB",
    "Macau": "MAC",
    "Kafia Kingi": None,

    # (optional but helpful)
    "Gambia, The": "GMB",
    "Macao": "MAC",
}

# extra synonyms to help pycountry (keep these)
PYCOUNTRY_SYNONYMS = {
    "Iran": "Iran, Islamic Republic of",
    "Syria": "Syrian Arab Republic",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Bolivia": "Bolivia, Plurinational State of",
    "United States": "United States of America",
    "United Kingdom": "United Kingdom of Great Britain and Northern Ireland",
    "Russia": "Russian Federation",
    "Tanzania": "Tanzania, United Republic of",
    "Vietnam": "Viet Nam",
    "Moldova": "Moldova, Republic of",
    "Laos": "Lao People's Democratic Republic",
    "Czech Republic": "Czechia",
    "Macau": "Macao",
}

# case-insensitive lookup helpers
OVERRIDE_TO_ISO3_CI = {k.casefold(): v for k, v in OVERRIDE_TO_ISO3.items()}
PYCOUNTRY_SYNONYMS_CI = {k.casefold(): v for k, v in PYCOUNTRY_SYNONYMS.items()}


def country_to_iso3(name) -> str | None:
    name = normalize_country_text(name)
    if not name:
        return None

    key = name.casefold()

    # 1) manual overrides first (case-insensitive)
    if key in OVERRIDE_TO_ISO3_CI:
        iso3 = OVERRIDE_TO_ISO3_CI[key]
        return iso3 if iso3 else None

    # 2) try pycountry direct (with synonyms)
    q = PYCOUNTRY_SYNONYMS_CI.get(key, name)
    try:
        hit = pycountry.countries.lookup(q)
        return hit.alpha_3
    except LookupError:
        # 3) optional fallback: fuzzy search (safe-ish)
        try:
            hit = pycountry.countries.search_fuzzy(q)[0]
            return hit.alpha_3
        except Exception:
            return None


# ---- apply to your dataframe ----
final = final.copy()

final["primary_country_clean"] = final["primary_country"].map(normalize_country_text)
final["primary_country_iso3"] = final["primary_country_clean"].map(country_to_iso3)

unmapped = (
    final.loc[
        final["primary_country_clean"].ne("") &
        final["primary_country_iso3"].isna(),
        "primary_country_clean",
    ]
    .value_counts()
    .head(50)
)

coverage_all = (
    final["primary_country_clean"].ne("") &
    final["primary_country_iso3"].notna()
).mean()

# Better coverage metric: only among "valid-looking" country strings
valid_mask = (
    final["primary_country_clean"].ne("") &
    ~final["primary_country_clean"].str.contains(
        "undetermined",
        case=False,
        na=False,
    )
)
coverage_valid = final.loc[valid_mask, "primary_country_iso3"].notna().mean()

print("Coverage (all rows, non-empty strings):", coverage_all)
print("Coverage (valid country strings only):", coverage_valid)
print("\nTop still-unmapped:")
print(unmapped)

Coverage (all rows, non-empty strings): 0.8286019780807271
Coverage (valid country strings only): 0.9996775025799793

Top still-unmapped:
primary_country_clean
undetermined            4
Kafia Kingi             4
Netherlands Antilles    1
Name: count, dtype: int64


In [ ]:
# Force these to be treated as "not a country"
FORCE_NULL = {"undetermined", "kafia kingi", "netherlands antilles"}

mask_force_null = final["primary_country_clean"].str.casefold().isin(FORCE_NULL)
final.loc[mask_force_null, "primary_country_iso3"] = None

# Optional: flag them for audit/debug
final["primary_country_iso3_note"] = ""
final.loc[mask_force_null, "primary_country_iso3_note"] = "non-country / deprecated"

In [ ]:
print("Still-unmapped valid countries:",
      final.loc[
          final["primary_country_clean"].ne("") &
          final["primary_country_iso3"].isna() &
          ~final["primary_country_clean"].str.casefold().isin(FORCE_NULL),
          "primary_country_clean"
      ].value_counts().head(20)
)

Still-unmapped valid countries: Series([], Name: count, dtype: int64)


In [ ]:
import os
import pandas as pd
import re

OUTDIR = "comtrade_oil"
os.makedirs(OUTDIR, exist_ok=True)

# --- If you already have: normalize_country_text() and country_to_iso3() defined, reuse them. ---

# 1) explode entity -> countries (list) to entity-country rows
ofac_links = (
    final[["ent_num", "countries"]]
    .explode("countries", ignore_index=True)
    .rename(columns={"countries": "country_raw"})
)

# 2) clean + map to ISO3
ofac_links["country_clean"] = ofac_links["country_raw"].map(normalize_country_text)
ofac_links["country_iso3"] = ofac_links["country_clean"].map(country_to_iso3)

# drop empties / invalids
ofac_links = ofac_links[
    ofac_links["country_iso3"].notna() &
    ofac_links["country_iso3"].astype(str).str.fullmatch(r"[A-Z]{3}")
].copy()

# optional: de-dup one entity-country link (so one SDN entity counted once per country)
ofac_links = ofac_links.drop_duplicates(subset=["ent_num", "country_iso3"])

# 3) aggregate to country-level
ofac_country_agg = (
    ofac_links.groupby("country_iso3", as_index=False)
    .agg(
        ofac_entities=("ent_num", "nunique"),   # how many SDN entities tied to that country
        ofac_entity_country_links=("ent_num", "size"),  # entity-country links (after dedup it's same as entities)
    )
    .sort_values(["ofac_entities"], ascending=False)
)

# 4) write outputs
ofac_links.to_csv(f"{OUTDIR}/ofac_entity_country_links.csv", index=False)
ofac_country_agg.to_csv(f"{OUTDIR}/ofac_country_agg.csv", index=False)

print("Wrote:", f"{OUTDIR}/ofac_country_agg.csv", "rows:", len(ofac_country_agg))
print(ofac_country_agg.head(15))

Wrote: comtrade_oil/ofac_country_agg.csv rows: 177
    country_iso3  ofac_entities  ofac_entity_country_links
135          RUS           5237                       5237
73           IRN           1519                       1519
103          MEX           1088                       1088
29           CHN           1072                       1072
4            ARE            668                        668
33           COL            383                        383
158          TUR            350                        350
90           LBN            331                        331
152          SYR            305                        305
19           BLR            260                        260
167          VEN            254                        254
162          UKR            242                        242
129          PRK            232                        232
108          MMR            191                        191
54           GBR            190                        190


In [ ]:
import pandas as pd

links = pd.read_csv("comtrade_oil/ofac_entity_country_links.csv")  # or whatever you named it

links["country_iso3"] = links["country_iso3"].astype(str).str.upper().str.strip()

ofac_country = (
    links.dropna(subset=["country_iso3", "ent_num"])
         .groupby("country_iso3")
         .agg(
             ofac_entities=("ent_num", "nunique"),
             ofac_entity_country_links=("ent_num", "size"),
         )
         .reset_index()
)

ofac_country.to_csv("comtrade_oil/ofac_country_agg.csv", index=False)
print(ofac_country.sort_values("ofac_entities", ascending=False).head(10))

    country_iso3  ofac_entities  ofac_entity_country_links
135          RUS           5237                       5237
73           IRN           1519                       1519
103          MEX           1088                       1088
29           CHN           1072                       1072
4            ARE            668                        668
33           COL            383                        383
158          TUR            350                        350
90           LBN            331                        331
152          SYR            305                        305
19           BLR            260                        260


In [ ]:
import pandas as pd

links = pd.read_csv("comtrade_oil/ofac_entity_country_links.csv")

print("rows links:", len(links))
print("unique entities:", links["ent_num"].nunique())
print("avg countries per entity:", len(links) / links["ent_num"].nunique())

# how many entities have >1 country?
multi = links.groupby("ent_num")["country_iso3"].nunique()
print("entities w/ >1 country:", (multi > 1).sum())

# top countries by links
print(links["country_iso3"].value_counts().head(15))

rows links: 16685
unique entities: 15504
avg countries per entity: 1.0761738906088751
entities w/ >1 country: 791
country_iso3
RUS    5237
IRN    1519
MEX    1088
CHN    1072
ARE     668
COL     383
TUR     350
LBN     331
SYR     305
BLR     260
VEN     254
UKR     242
PRK     232
MMR     191
GBR     190
Name: count, dtype: int64


In [ ]:
import os
import pandas as pd

OUTDIR = "comtrade_oil"
os.makedirs(OUTDIR, exist_ok=True)

nodes = pd.read_csv(f"{OUTDIR}/country_nodes.csv")
ofac  = pd.read_csv(f"{OUTDIR}/ofac_country_agg.csv")

# ---------- normalize / standardize column names ----------
# nodes: accept iso3 or country_iso3
if "country_iso3" not in nodes.columns:
    if "iso3" in nodes.columns:
        nodes = nodes.rename(columns={"iso3": "country_iso3"})
    else:
        raise KeyError(
            f"Nodes file missing ISO3 column. Columns are: {nodes.columns.tolist()}"
        )

# nodes: accept name variants -> country_name (THIS FIXES YOUR DROP)
if "country_name" not in nodes.columns:
    for cand in ["name", "country", "reporterDesc", "partnerDesc"]:
        if cand in nodes.columns:
            nodes = nodes.rename(columns={cand: "country_name"})
            break

# ofac: accept iso3 or country_iso3
if "country_iso3" not in ofac.columns:
    if "iso3" in ofac.columns:
        ofac = ofac.rename(columns={"iso3": "country_iso3"})
    else:
        raise KeyError(
            f"OFAC agg missing country_iso3. Columns are: {ofac.columns.tolist()}"
        )

# ---------- clean keys ----------
nodes["country_iso3"] = nodes["country_iso3"].astype(str).str.upper().str.strip()
ofac["country_iso3"]  = ofac["country_iso3"].astype(str).str.upper().str.strip()

# keep only valid ISO3 rows in nodes
nodes = nodes[nodes["country_iso3"].str.len() == 3].copy()

# --- optional: clean names (so mode_value is stable)
if "country_name" in nodes.columns:
    nodes["country_name"] = nodes["country_name"].astype(str).str.strip()
    nodes.loc[nodes["country_name"].isin(["", "nan", "None"]), "country_name"] = pd.NA

# ---------- check duplicates before dedupe ----------
dups = nodes[nodes.duplicated("country_iso3", keep=False)].sort_values("country_iso3")
print("Duplicate ISO3 rows in nodes:", len(dups))
if len(dups) > 0:
    cols_show = [c for c in
                 ["country_iso3", "code", "country_name", "iso2", "is_group"]
                 if c in nodes.columns]
    print(dups[cols_show].head(30))

def mode_value(s: pd.Series):
    s = s.dropna()
    if len(s) == 0:
        return None
    return s.value_counts().index[0]

# ---------- dedupe nodes: 1 row per ISO3 ----------
agg_dict = {}

if "code" in nodes.columns:
    agg_dict["code"] = ("code", "min")
if "country_name" in nodes.columns:
    agg_dict["country_name"] = ("country_name", mode_value)
if "iso2" in nodes.columns:
    agg_dict["iso2"] = ("iso2", mode_value)
if "is_group" in nodes.columns:
    # ensure bool-ish before max
    nodes["is_group"] = nodes["is_group"].astype(str).str.lower().isin(["true", "1", "yes"])
    agg_dict["is_group"] = ("is_group", "max")

# If your nodes file only has ISO3 + name, this is still fine.
nodes_u = nodes.groupby("country_iso3", as_index=False).agg(**agg_dict)

if not nodes_u["country_iso3"].is_unique:
    raise ValueError("Still not unique after dedupe (unexpected).")

# ---------- validate OFAC keys unique ----------
if not ofac["country_iso3"].is_unique:
    bad = ofac[ofac.duplicated("country_iso3", keep=False)].sort_values("country_iso3")
    raise ValueError(f"OFAC agg has duplicate ISO3 keys. Example:\n{bad.head(20)}")

# ---------- merge ----------
nodes_plus = nodes_u.merge(ofac, on="country_iso3", how="left")

for c in ["ofac_entities", "ofac_entity_country_links"]:
    if c in nodes_plus.columns:
        nodes_plus[c] = nodes_plus[c].fillna(0).astype(int)

out_path = f"{OUTDIR}/country_nodes_with_ofac.csv"
nodes_plus.to_csv(out_path, index=False)

# ---------- coverage ----------
if "ofac_entities" in nodes_plus.columns:
    cov_all = (nodes_plus["ofac_entities"] > 0).mean()
    if "is_group" in nodes_plus.columns:
        cov_countries = (nodes_plus.loc[~nodes_plus["is_group"], "ofac_entities"] > 0).mean()
    else:
        cov_countries = cov_all
else:
    cov_all = cov_countries = 0.0

print("Wrote:", out_path)
print("Nodes (deduped):", len(nodes_u))
print("Coverage (all nodes):", cov_all)
print("Coverage (non-group only):", cov_countries)
print(nodes_plus.sort_values("ofac_entities", ascending=False).head(15))

Duplicate ISO3 rows in nodes: 440
    country_iso3  code          country_name iso2  is_group
397          A79   636  Rest of America, nes  NaN     False
339          A79   473             LAIA, nes  NaN     False
231          ABW   533                 Aruba   AW     False
9            ABW   533                 Aruba   AW     False
0            AFG     4           Afghanistan   AF     False
219          AFG     4           Afghanistan   AF     False
4            AGO    24                Angola   AO     False
224          AGO    24                Angola   AO     False
225          AIA   660              Anguilla   AI     False
5            AIA   660              Anguilla   AI     False
1            ALB     8               Albania   AL     False
220          ALB     8               Albania   AL     False
223          AND    20               Andorra   AD     False
3            AND    20               Andorra   AD     False
447          ARE   784  United Arab Emirates   AE     False
206   

In [ ]:
import re
import pandas as pd

edges = pd.read_csv("comtrade_oil/country_edges_oil_2019plus.csv")

for c in ["exporter_iso3", "importer_iso3"]:
    edges[c] = edges[c].astype(str).str.upper().str.strip()

iso3_pat = re.compile(r"^[A-Z]{3}$")
good = edges["exporter_iso3"].str.match(iso3_pat) & edges["importer_iso3"].str.match(iso3_pat)

print("dropping non-ISO3 edges:", (~good).sum())
print("top bad importers:", edges.loc[~good, "importer_iso3"].value_counts().head(20))

edges = edges.loc[good].copy()
edges.to_csv("comtrade_oil/country_edges_oil_2019plus_clean.csv", index=False)
print("wrote clean edges:", edges.shape)

dropping non-ISO3 edges: 1567
top bad importers: importer_iso3
_X     355
S19    338
X1     143
E19     56
X2      39
F19     31
XX      15
A79     10
MEX      6
PAK      6
PLW      6
MHL      6
FSM      6
NOR      6
NZL      6
MOZ      6
MDG      6
MUS      6
MYS      6
PNG      6
Name: count, dtype: int64
wrote clean edges: (45513, 6)


In [ ]:
import pandas as pd

OUTDIR = "comtrade_oil"

nodes = pd.read_csv(f"{OUTDIR}/country_nodes_with_ofac.csv")
edges = pd.read_csv(f"{OUTDIR}/country_edges_oil_2019plus_clean.csv")

print("nodes:", nodes.shape)
print("edges:", edges.shape)

# 1) node keys unique
assert nodes["country_iso3"].is_unique

# 2) edges only refer to known nodes
node_set = set(nodes["country_iso3"])
bad_exp = set(edges["exporter_iso3"]) - node_set
bad_imp = set(edges["importer_iso3"]) - node_set
print("bad exporters:", len(bad_exp), list(sorted(bad_exp))[:20])
print("bad importers:", len(bad_imp), list(sorted(bad_imp))[:20])

# 3) years look right
print("years:", sorted(edges["year"].unique()))

# 4) OFAC columns exist + are ints
for c in ["ofac_entities", "ofac_entity_country_links"]:
    assert c in nodes.columns
    print(c, nodes[c].dtype, "nonzero:", (nodes[c] > 0).sum())

nodes: (238, 7)
edges: (45513, 6)
bad exporters: 0 []
bad importers: 0 []
years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
ofac_entities int64 nonzero: 171
ofac_entity_country_links int64 nonzero: 171


In [ ]:
import pandas as pd
import re

nodes = pd.read_csv("comtrade_oil/country_nodes_with_ofac.csv")
edges  = pd.read_csv("comtrade_oil/country_edges_oil_2019plus_clean.csv")

# normalize
nodes["country_iso3"] = nodes["country_iso3"].astype(str).str.upper().str.strip()
edges["exporter_iso3"] = edges["exporter_iso3"].astype(str).str.upper().str.strip()
edges["importer_iso3"] = edges["importer_iso3"].astype(str).str.upper().str.strip()

node_set = set(nodes["country_iso3"].dropna())
edge_set = set(edges["exporter_iso3"]).union(set(edges["importer_iso3"]))

missing = sorted(edge_set - node_set)
print("unique nodes:", len(node_set))
print("unique iso3 in edges:", len(edge_set))
print("missing iso3 in nodes:", len(missing))
print("sample missing:", missing[:30])

unique nodes: 238
unique iso3 in edges: 232
missing iso3 in nodes: 0
sample missing: []


In [ ]:
import os
import re
import pandas as pd

OUTDIR = "comtrade_oil/final"
os.makedirs(OUTDIR, exist_ok=True)

nodes = pd.read_csv("comtrade_oil/country_nodes_with_ofac.csv")
edges = pd.read_csv("comtrade_oil/country_edges_oil_2019plus_clean.csv")

# -------------------------
# Normalize ISO3
# -------------------------
def norm_iso3(s):
    return s.astype(str).str.upper().str.strip()

nodes["country_iso3"] = norm_iso3(nodes["country_iso3"])
edges["exporter_iso3"] = norm_iso3(edges["exporter_iso3"])
edges["importer_iso3"] = norm_iso3(edges["importer_iso3"])

# Keep only valid ISO3 (strict A-Z{3})
iso3_re = re.compile(r"^[A-Z]{3}$")
nodes = nodes[nodes["country_iso3"].map(lambda x: bool(iso3_re.match(x)))].copy()
edges = edges[
    edges["exporter_iso3"].map(lambda x: bool(iso3_re.match(x))) &
    edges["importer_iso3"].map(lambda x: bool(iso3_re.match(x)))
].copy()

# -------------------------
# Trim nodes to what edges use
# -------------------------
edge_iso3 = set(edges["exporter_iso3"]).union(set(edges["importer_iso3"]))
nodes = nodes[nodes["country_iso3"].isin(edge_iso3)].copy()

# Dedupe nodes: one row per ISO3
nodes = nodes.sort_values(["country_iso3"]).drop_duplicates("country_iso3", keep="first")

# -------------------------
# Standardize / coerce types
# -------------------------
# (keep these if present)
for c in ["ofac_entities", "ofac_entity_country_links"]:
    if c in nodes.columns:
        nodes[c] = pd.to_numeric(nodes[c], errors="coerce").fillna(0).astype(int)

if "trade_value_usd" in edges.columns:
    edges["trade_value_usd"] = pd.to_numeric(edges["trade_value_usd"], errors="coerce").fillna(0.0)

if "year" in edges.columns:
    edges["year"] = pd.to_numeric(edges["year"], errors="coerce").astype("Int64")

# Drop self-loops (optional, but usually cleaner for “hub” analysis)
edges = edges[edges["exporter_iso3"] != edges["importer_iso3"]].copy()

# If you still have duplicates per (year, exporter, importer), aggregate
key = ["year", "exporter_iso3", "importer_iso3"]
if all(k in edges.columns for k in key) and "trade_value_usd" in edges.columns:
    edges = (
        edges.groupby(key, as_index=False)["trade_value_usd"]
        .sum()
    )

# -------------------------
# Final column order
# -------------------------
nodes_cols = [c for c in [
    "country_iso3", "country_name", "iso2", "code", "is_group",
    "ofac_entities", "ofac_entity_country_links"
] if c in nodes.columns]
nodes = nodes[nodes_cols].copy()

edges_cols = [c for c in ["year", "exporter_iso3", "importer_iso3", "trade_value_usd"] if c in edges.columns]
edges = edges[edges_cols].copy()

# -------------------------
# Write final outputs
# -------------------------
nodes_out = f"{OUTDIR}/nodes_country_oil_2019plus.csv"
edges_out = f"{OUTDIR}/edges_country_oil_2019plus.csv"
nodes.to_csv(nodes_out, index=False)
edges.to_csv(edges_out, index=False)

print("WROTE:", nodes_out, "rows:", len(nodes), "unique iso3:", nodes["country_iso3"].nunique())
print("WROTE:", edges_out, "rows:", len(edges))
print("years:", sorted([y for y in edges["year"].dropna().unique().tolist()]) if "year" in edges.columns else "no year col")

WROTE: comtrade_oil/final/nodes_country_oil_2019plus.csv rows: 232 unique iso3: 232
WROTE: comtrade_oil/final/edges_country_oil_2019plus.csv rows: 45513
years: [2019, 2020, 2021, 2022, 2023, 2024]


In [ ]:
import pandas as pd
import re

nodes = pd.read_csv("comtrade_oil/final/nodes_country_oil_2019plus.csv")
edges = pd.read_csv("comtrade_oil/final/edges_country_oil_2019plus.csv")

iso3_re = re.compile(r"^[A-Z]{3}$")
assert nodes["country_iso3"].map(lambda x: bool(iso3_re.match(str(x)))).all()
assert edges["exporter_iso3"].map(lambda x: bool(iso3_re.match(str(x)))).all()
assert edges["importer_iso3"].map(lambda x: bool(iso3_re.match(str(x)))).all()

node_set = set(nodes["country_iso3"])
edge_set = set(edges["exporter_iso3"]).union(set(edges["importer_iso3"]))
print("missing iso3 in nodes:", len(edge_set - node_set))
print("nodes:", nodes.shape, "edges:", edges.shape)
print("duplicate nodes:", nodes["country_iso3"].duplicated().sum())
print("duplicate edges:", edges.duplicated().sum())

missing iso3 in nodes: 0
nodes: (232, 6) edges: (45513, 4)
duplicate nodes: 0
duplicate edges: 0


In [ ]:
import os
import pandas as pd

FINAL_NODES = "comtrade_oil/final/nodes_country_oil_2019plus.csv"
FINAL_EDGES = "comtrade_oil/final/edges_country_oil_2019plus.csv"
OFAC_AGG = "comtrade_oil/ofac_country_agg.csv"
HUBS = "comtrade_oil/shadow_hubs_residual_2019plus.csv"

for p in [FINAL_NODES, FINAL_EDGES, OFAC_AGG, HUBS]:
    print(p, "exists:", os.path.exists(p))

nodes = pd.read_csv(FINAL_NODES)
edges = pd.read_csv(FINAL_EDGES)

req_node_cols = {"country_iso3", "country_name",
                 "ofac_entities", "ofac_entity_country_links"}
req_edge_cols = {"exporter_iso3", "importer_iso3",
                 "year", "trade_value_usd"}

print("nodes shape:", nodes.shape)
print("edges shape:", edges.shape)
print("missing node cols:", req_node_cols - set(nodes.columns))
print("missing edge cols:", req_edge_cols - set(edges.columns))

node_set = set(nodes["country_iso3"].astype(str))
edge_set = set(edges["exporter_iso3"].astype(str)) | \
           set(edges["importer_iso3"].astype(str))

missing = sorted(edge_set - node_set)
print("iso3 in edges not in nodes:", len(missing))
print("sample missing:", missing[:30])

print("years:", sorted(edges["year"].unique()))
print("null trade_value_usd:", edges["trade_value_usd"].isna().sum())
print("nonpositive trade_value_usd:",
      (edges["trade_value_usd"] <= 0).sum())

comtrade_oil/final/nodes_country_oil_2019plus.csv exists: True
comtrade_oil/final/edges_country_oil_2019plus.csv exists: True
comtrade_oil/ofac_country_agg.csv exists: True
comtrade_oil/shadow_hubs_residual_2019plus.csv exists: True
nodes shape: (232, 7)
edges shape: (45513, 4)
missing node cols: set()
missing edge cols: set()
iso3 in edges not in nodes: 0
sample missing: []
years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
null trade_value_usd: 0
nonpositive trade_value_usd: 0


In [ ]:
import pandas as pd

nodes_path = "comtrade_oil/final/nodes_country_oil_2019plus.csv"
nodes = pd.read_csv(nodes_path)

print("Before:", nodes.columns.tolist())

if "country_name" not in nodes.columns and "name" in nodes.columns:
    nodes = nodes.rename(columns={"name": "country_name"})

# Ensure ISO3 is clean
nodes["country_iso3"] = nodes["country_iso3"].astype(str).str.upper().str.strip()

nodes.to_csv(nodes_path, index=False)

print("After:", nodes.columns.tolist())
print("Wrote:", nodes_path, "rows:", len(nodes))

Before: ['country_iso3', 'country_name', 'iso2', 'code', 'is_group', 'ofac_entities', 'ofac_entity_country_links']
After: ['country_iso3', 'country_name', 'iso2', 'code', 'is_group', 'ofac_entities', 'ofac_entity_country_links']
Wrote: comtrade_oil/final/nodes_country_oil_2019plus.csv rows: 232


In [ ]:
import os
import pandas as pd
import numpy as np

FILES = {
    "nodes_final": "comtrade_oil/final/nodes_country_oil_2019plus.csv",
    "edges_final": "comtrade_oil/final/edges_country_oil_2019plus.csv",
    "ofac_agg": "comtrade_oil/ofac_country_agg.csv",
    "shadow_hubs": "comtrade_oil/shadow_hubs_residual_2019plus.csv",
}

def show_file(path: str, n: int = 10):
    print("\n" + "=" * 90)
    print("FILE:", path)
    print("exists:", os.path.exists(path))
    if not os.path.exists(path):
        return None

    df = pd.read_csv(path)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("\nhead():")
    display(df.head(n))

    # lightweight checks
    for c in ["country_iso3", "exporter_iso3", "importer_iso3", "iso3"]:
        if c in df.columns:
            bad = df[c].astype(str).str.strip().str.len().ne(3).sum()
            print(f"bad ISO3 len in {c}:", int(bad))

    if "trade_value_usd" in df.columns:
        tv = pd.to_numeric(df["trade_value_usd"], errors="coerce")
        print("trade_value_usd null:", int(tv.isna().sum()))
        print("trade_value_usd <= 0:", int((tv <= 0).sum()))
        print("trade_value_usd min/median/max:",
              float(tv.min()), float(tv.median()), float(tv.max()))

    if "year" in df.columns:
        y = pd.to_numeric(df["year"], errors="coerce")
        print("years:", sorted([int(x) for x in y.dropna().unique()])[:20], "...")

    return df

dfs = {}
for k, p in FILES.items():
    dfs[k] = show_file(p, n=10)

# extra: nodes vs edges coverage (ISO3 in edges should exist in nodes)
if dfs.get("nodes_final") is not None and dfs.get("edges_final") is not None:
    nodes = dfs["nodes_final"]
    edges = dfs["edges_final"]

    # pick node iso3 col
    node_iso_col = "country_iso3" if "country_iso3" in nodes.columns else ("iso3" if "iso3" in nodes.columns else None)
    if node_iso_col is None:
        print("\n[WARN] Could not find ISO3 column in nodes.")
    else:
        node_set = set(nodes[node_iso_col].dropna().astype(str).str.strip().str.upper())
        exp_col = "exporter_iso3" if "exporter_iso3" in edges.columns else None
        imp_col = "importer_iso3" if "importer_iso3" in edges.columns else None

        if exp_col and imp_col:
            edge_set = set(edges[exp_col].astype(str).str.strip().str.upper()) | set(edges[imp_col].astype(str).str.strip().str.upper())
            missing = sorted(edge_set - node_set)
            print("\n" + "-" * 90)
            print("ISO3 in edges not in nodes:", len(missing))
            print("sample missing:", missing[:30])
        else:
            print("\n[WARN] Could not find exporter/importer ISO3 columns in edges.")


FILE: comtrade_oil/final/nodes_country_oil_2019plus.csv
exists: True
shape: (232, 7)
columns: ['country_iso3', 'country_name', 'iso2', 'code', 'is_group', 'ofac_entities', 'ofac_entity_country_links']

head():


,country_iso3,country_name,iso2,code,is_group,ofac_entities,ofac_entity_country_links
0,ABW,Aruba,AW,533,False,1,1
1,AFG,Afghanistan,AF,4,False,82,82
2,AGO,Angola,AO,24,False,11,11
3,AIA,Anguilla,AI,660,False,0,0
4,ALB,Albania,AL,8,False,8,8
5,AND,Andorra,AD,20,False,0,0
6,ARE,United Arab Emirates,AE,784,False,668,668
7,ARG,Argentina,AR,32,False,19,19
8,ARM,Armenia,AM,51,False,6,6
9,ASM,American Samoa,AS,16,False,0,0


bad ISO3 len in country_iso3: 0

FILE: comtrade_oil/final/edges_country_oil_2019plus.csv
exists: True
shape: (45513, 4)
columns: ['year', 'exporter_iso3', 'importer_iso3', 'trade_value_usd']

head():


,year,exporter_iso3,importer_iso3,trade_value_usd
0,2019,ABW,COL,3.910600e+01
1,2019,ABW,CUW,2.724022e+03
2,2019,ABW,NLD,5.474900e+01
3,2019,ABW,PAN,3.299441e+03
4,2019,ABW,USA,3.547490e+02
5,2019,ABW,VEN,2.793300e+01
6,2019,AGO,ARE,1.066114e+08
7,2019,AGO,BEL,8.942041e+07
8,2019,AGO,CHL,3.470844e+08
9,2019,AGO,CHN,4.282477e+10


bad ISO3 len in exporter_iso3: 0
bad ISO3 len in importer_iso3: 0
trade_value_usd null: 0
trade_value_usd <= 0: 0
trade_value_usd min/median/max: 0.001 835989.0 530956549149.51
years: [2019, 2020, 2021, 2022, 2023, 2024] ...

FILE: comtrade_oil/ofac_country_agg.csv
exists: True
shape: (177, 3)
columns: ['country_iso3', 'ofac_entities', 'ofac_entity_country_links']

head():


,country_iso3,ofac_entities,ofac_entity_country_links
0,ABW,1,1
1,AFG,82,82
2,AGO,11,11
3,ALB,8,8
4,ARE,668,668
5,ARG,19,19
6,ARM,6,6
7,ATG,2,2
8,AUS,6,6
9,AUT,15,15


bad ISO3 len in country_iso3: 0

FILE: comtrade_oil/shadow_hubs_residual_2019plus.csv
exists: True
shape: (264, 14)
columns: ['year', 'iso3', 'betweenness', 'trade_total_usd', 'in_deg', 'out_deg', 'prune_threshold_usd', 'edges_after_pruning', 'nodes_after_pruning', 'log_btw', 'log_vol', 'btw_expected', 'shadow_resid', 'shadow_rank']

head():


,year,iso3,betweenness,trade_total_usd,in_deg,out_deg,prune_threshold_usd,edges_after_pruning,nodes_after_pruning,log_btw,log_vol,btw_expected,shadow_resid,shadow_rank
0,2019,ARE,0.050841,2.028978e+11,41,50,10464948.34,2434,218,0.049590,26.035968,0.080311,-0.030721,39.0
1,2019,BEL,0.004608,4.981715e+10,32,46,10464948.34,2434,218,0.004598,24.631625,0.042939,-0.038342,44.0
2,2019,CHN,0.102001,2.641658e+11,54,65,10464948.34,2434,218,0.097128,26.299843,0.087333,0.009795,15.0
3,2019,ESP,0.152884,1.772770e+11,44,83,10464948.34,2434,218,0.142267,25.900979,0.076719,0.065548,4.0
4,2019,FRA,0.027500,8.869057e+10,46,74,10464948.34,2434,218,0.027129,25.208419,0.058289,-0.031160,40.0
5,2019,GBR,0.079642,2.076532e+11,46,47,10464948.34,2434,218,0.076630,26.059135,0.080928,-0.004298,23.0
6,2019,IND,0.145929,9.129588e+10,41,57,10464948.34,2434,218,0.136216,25.237371,0.059059,0.077157,2.0
7,2019,ITA,0.028034,1.257726e+11,39,54,10464948.34,2434,218,0.027648,25.557741,0.067585,-0.039937,45.0
8,2019,JPN,0.002667,3.626153e+10,32,26,10464948.34,2434,218,0.002663,24.314023,0.034487,-0.031824,41.0
9,2019,KOR,0.047256,8.206678e+10,35,46,10464948.34,2434,218,0.046174,25.130799,0.056223,-0.010049,27.0


bad ISO3 len in iso3: 0
years: [2019, 2020, 2021, 2022, 2023, 2024] ...

------------------------------------------------------------------------------------------
ISO3 in edges not in nodes: 0
sample missing: []


In [ ]:
import pandas as pd

NODES_PATH = "comtrade_oil/final/nodes_country_oil_2019plus.csv"
EDGES_PATH = "comtrade_oil/final/edges_country_oil_2019plus.csv"
OFAC_PATH  = "comtrade_oil/ofac_country_agg.csv"
SHADOW_PATH = "comtrade_oil/shadow_hubs_residual_2019plus.csv"

nodes = pd.read_csv(NODES_PATH)
edges = pd.read_csv(EDGES_PATH)
ofac  = pd.read_csv(OFAC_PATH)
shadow = pd.read_csv(SHADOW_PATH)

print("nodes:", nodes.shape)
display(nodes.head(10))

print("edges:", edges.shape)
display(edges.head(10))

print("ofac:", ofac.shape)
display(ofac.head(10))

print("shadow:", shadow.shape)
display(shadow.head(10))


nodes: (232, 7)


,country_iso3,country_name,iso2,code,is_group,ofac_entities,ofac_entity_country_links
0,ABW,Aruba,AW,533,False,1,1
1,AFG,Afghanistan,AF,4,False,82,82
2,AGO,Angola,AO,24,False,11,11
3,AIA,Anguilla,AI,660,False,0,0
4,ALB,Albania,AL,8,False,8,8
5,AND,Andorra,AD,20,False,0,0
6,ARE,United Arab Emirates,AE,784,False,668,668
7,ARG,Argentina,AR,32,False,19,19
8,ARM,Armenia,AM,51,False,6,6
9,ASM,American Samoa,AS,16,False,0,0


edges: (45513, 4)


,year,exporter_iso3,importer_iso3,trade_value_usd
0,2019,ABW,COL,3.910600e+01
1,2019,ABW,CUW,2.724022e+03
2,2019,ABW,NLD,5.474900e+01
3,2019,ABW,PAN,3.299441e+03
4,2019,ABW,USA,3.547490e+02
5,2019,ABW,VEN,2.793300e+01
6,2019,AGO,ARE,1.066114e+08
7,2019,AGO,BEL,8.942041e+07
8,2019,AGO,CHL,3.470844e+08
9,2019,AGO,CHN,4.282477e+10


ofac: (177, 3)


,country_iso3,ofac_entities,ofac_entity_country_links
0,ABW,1,1
1,AFG,82,82
2,AGO,11,11
3,ALB,8,8
4,ARE,668,668
5,ARG,19,19
6,ARM,6,6
7,ATG,2,2
8,AUS,6,6
9,AUT,15,15


shadow: (413, 14)


,year,iso3,betweenness,trade_total_usd,in_deg,out_deg,prune_threshold_usd,edges_after_pruning,nodes_after_pruning,log_btw,log_vol,btw_expected,shadow_resid,shadow_rank
0,2019,COL,0.011684,6.830752e+10,45,51,0.001,8111,237,0.011616,24.947286,0.055398,-0.043782,70.0
1,2019,NLD,0.201190,2.235856e+11,107,184,0.001,8111,237,0.183313,26.133060,0.079287,0.104025,3.0
2,2019,PAN,0.004237,2.495475e+10,63,24,0.001,8111,237,0.004228,23.940330,0.035111,-0.030883,60.0
3,2019,USA,0.329463,6.010105e+11,115,181,0.001,8111,237,0.284775,27.121878,0.099209,0.185566,1.0
4,2019,AGO,0.002939,6.409265e+10,45,28,0.001,8111,237,0.002935,24.883595,0.054115,-0.051180,72.0
5,2019,ARE,0.078038,2.031445e+11,89,158,0.001,8111,237,0.075143,26.037183,0.077356,-0.002213,34.0
6,2019,BEL,0.004237,5.012300e+10,81,183,0.001,8111,237,0.004228,24.637746,0.049162,-0.044934,71.0
7,2019,CHN,0.157843,2.643231e+11,87,174,0.001,8111,237,0.146559,26.300438,0.082660,0.063900,7.0
8,2019,ESP,0.174811,1.775064e+11,84,148,0.001,8111,237,0.161107,25.902272,0.074638,0.086469,4.0
9,2019,FRA,0.050126,8.897249e+10,100,184,0.001,8111,237,0.048910,25.211593,0.060723,-0.011813,36.0


<a name='auradb' />

# Graph Database (Neo4j AuraDB)

<a href=#top>↑ Return to Top of Notebook</a>

The cleaned CSV datasets were loaded into a **Neo4j AuraDB** (free tier) graph database for structured Cypher queries and downstream GraphRAG.

**Data model:**

| Element | Description |
|---------|------------|
| `(:Country)` nodes | Keyed by `iso3` — name, iso2, comtrade_code, ofac_entities, ofac_links |
| `[:TRADE]` edges | `(:Country)-[:TRADE {year, trade_value_usd}]->(:Country)` |
| `(:Year)` nodes | One per year (2019–2024) |
| `[:SHADOW_HUB]` edges | `(:Country)-[:SHADOW_HUB {betweenness, shadow_resid, shadow_rank, ...}]->(:Year)` |

Load scripts: [`auradb_load.cypher`](./auradb_load.cypher)


In [ ]:
# Install neo4j driver
!pip install neo4j python-dotenv -q

import os
from neo4j import GraphDatabase
import pandas as pd

# ── Connection ─────────────────────────────────────────────
# Copy .env.example → .env and fill in your credentials
from dotenv import load_dotenv
load_dotenv()

URI  = os.environ["NEO4J_URI"]
AUTH = (os.environ.get("NEO4J_USERNAME", "neo4j"), os.environ["NEO4J_PASSWORD"])

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("✅ Connected to AuraDB instance")


In [ ]:
# ── Load Statistics (queried live from AuraDB) ─────────────
with driver.session() as session:
    stats = {}

    # Count Country nodes
    r = session.run("MATCH (c:Country) WHERE NOT c:RegionGroup RETURN count(c) AS n")
    stats["Country nodes"] = r.single()["n"]

    # Count Year nodes
    r = session.run("MATCH (y:Year) RETURN count(y) AS n")
    stats["Year nodes"] = r.single()["n"]

    # Count TRADE relationships
    r = session.run("MATCH ()-[t:TRADE]->() RETURN count(t) AS n")
    stats["TRADE relationships"] = r.single()["n"]

    # Count SHADOW_HUB relationships
    r = session.run("MATCH ()-[s:SHADOW_HUB]->() RETURN count(s) AS n")
    stats["SHADOW_HUB relationships"] = r.single()["n"]

stats_df = pd.DataFrame(list(stats.items()), columns=["Element", "Count"])
stats_df.loc[len(stats_df)] = ["Total", stats_df["Count"].sum()]
print("AuraDB Load Statistics")
print("=" * 40)
stats_df


In [ ]:
# ── Validation: Cross-reference AuraDB against source CSVs ──
import os

DATA_DIR = "../data"
nodes_csv = pd.read_csv(os.path.join(DATA_DIR, "nodes_country_oil_2019plus.csv"))
edges_csv = pd.read_csv(os.path.join(DATA_DIR, "edges_country_oil_2019plus.csv"))
shadow_csv = pd.read_csv(os.path.join(DATA_DIR, "shadow_hubs_residual_2019plus.csv"))

tests_passed = 0
tests_total = 0

# Test 1: Country node count matches CSV (excluding groups)
csv_countries = nodes_csv[nodes_csv["is_group"] == False].shape[0]
tests_total += 1
if stats["Country nodes"] == csv_countries:
    print(f"✅ Test 1: Country count matches CSV ({csv_countries})")
    tests_passed += 1
else:
    print(f"❌ Test 1: Country count mismatch — AuraDB: {stats['Country nodes']}, CSV: {csv_countries}")

# Test 2: TRADE edge count matches CSV
csv_edges = edges_csv.shape[0]
tests_total += 1
if stats["TRADE relationships"] == csv_edges:
    print(f"✅ Test 2: TRADE edge count matches CSV ({csv_edges})")
    tests_passed += 1
else:
    print(f"❌ Test 2: TRADE edge count mismatch — AuraDB: {stats['TRADE relationships']}, CSV: {csv_edges}")

# Test 3: SHADOW_HUB count matches CSV
csv_shadow = shadow_csv.shape[0]
tests_total += 1
if stats["SHADOW_HUB relationships"] == csv_shadow:
    print(f"✅ Test 3: SHADOW_HUB count matches CSV ({csv_shadow})")
    tests_passed += 1
else:
    print(f"❌ Test 3: SHADOW_HUB count mismatch — AuraDB: {stats['SHADOW_HUB relationships']}, CSV: {csv_shadow}")

# Test 4: Year coverage matches
csv_years = sorted(edges_csv["year"].unique())
with driver.session() as session:
    r = session.run("MATCH (y:Year) RETURN y.year AS year ORDER BY year")
    db_years = sorted([rec["year"] for rec in r])
tests_total += 1
if csv_years == db_years:
    print(f"✅ Test 4: Year coverage matches ({csv_years})")
    tests_passed += 1
else:
    print(f"❌ Test 4: Year mismatch — AuraDB: {db_years}, CSV: {csv_years}")

# Test 5: ISO3 integrity — all edge ISO3s exist as nodes
with driver.session() as session:
    r = session.run("""
        MATCH (exp:Country)-[t:TRADE]->(imp:Country)
        WITH collect(DISTINCT exp.iso3) + collect(DISTINCT imp.iso3) AS trade_isos
        UNWIND trade_isos AS iso
        WITH collect(DISTINCT iso) AS trade_set
        MATCH (c:Country)
        WITH trade_set, collect(c.iso3) AS node_set
        RETURN size([x IN trade_set WHERE NOT x IN node_set]) AS orphans
    """)
    orphans = r.single()["orphans"]
tests_total += 1
if orphans == 0:
    print(f"✅ Test 5: ISO3 integrity — all trade ISO3s exist as nodes")
    tests_passed += 1
else:
    print(f"❌ Test 5: {orphans} orphan ISO3 codes in TRADE edges")

# Test 6: Spot-check a specific value (USA 2024 shadow rank)
with driver.session() as session:
    r = session.run("""
        MATCH (c:Country {iso3: 'USA'})-[s:SHADOW_HUB]->(y:Year {year: 2024})
        RETURN s.shadow_rank AS rank, s.shadow_resid AS resid
    """)
    rec = r.single()
csv_usa = shadow_csv[(shadow_csv["iso3"] == "USA") & (shadow_csv["year"] == 2024)]
tests_total += 1
if abs(rec["rank"] - csv_usa["shadow_rank"].values[0]) < 0.001:
    print(f"✅ Test 6: USA 2024 shadow_rank spot check passes (rank={rec['rank']})")
    tests_passed += 1
else:
    print(f"❌ Test 6: USA 2024 shadow_rank mismatch")

print(f"\n{'=' * 40}")
print(f"Results: {tests_passed}/{tests_total} tests passed")


In [ ]:
# ── Demo: Top 10 Shadow Hubs in 2024 with OFAC context ──────
with driver.session() as session:
    result = session.run("""
        MATCH (c:Country)-[s:SHADOW_HUB]->(y:Year {year: 2024})
        RETURN c.iso3 AS iso3, c.name AS country,
               round(s.shadow_resid, 4) AS residual,
               s.shadow_rank AS rank,
               round(s.betweenness, 4) AS betweenness,
               round(s.trade_total_usd / 1e9, 2) AS trade_volume_B,
               c.ofac_entities AS ofac_entities
        ORDER BY s.shadow_rank ASC
        LIMIT 10
    """)
    top_hubs = pd.DataFrame([dict(r) for r in result])

print("Top 10 Shadow Hubs (2024) — queried from AuraDB")
top_hubs



## Next Steps

<a href=#top>↑ Return to Top of Notebook</a>

**Network Analyst:** The graph is ready for Cypher-based centrality and community queries. Compute weighted in/out strength directly in Cypher, or pull data into NetworkX via the `neo4j` Python driver. See demo queries Q1–Q8 in [`auradb_load.cypher`](./auradb_load.cypher).

**AI Engineer / GraphRAG:** Connect using the driver setup above. The `SHADOW_HUB` relationship properties give you a clean structured "facts table" — every answer your RAG pipeline produces should cite specific graph properties.

**AuraDB Connection:** Copy `.env.example` to `.env` and fill in your credentials. The instance auto-pauses after 72 hours of inactivity — resume from [console.neo4j.io](https://console.neo4j.io).


<a name='globe' />

# Interactive Globe Visualization

<a href=#top>↑ Return to Top of Notebook</a>

A standalone 3D globe visualization ([`globe_viz.html`](./globe_viz.html)) displays oil trade flows geographically. Open in any browser.

**How to use:** Click any country dot to reveal its trade flow arcs. Click again or click the globe background to deselect. Use the year dropdown to switch between 2019–2024.

**Visual encoding:**
- Dot color = OFAC sanctions exposure (red > orange > blue)
- Dot size = shadow hub residual rank (larger = more anomalously central)
- Arc color/width = trade value (gold = highest)

The visualization embeds data directly and does not query AuraDB.


<a name='analytics' />

# Network Analytics

<a href=#top>↑ Return to Top of Notebook</a>

*Placeholder — centrality metrics, community detection, weighted degree analysis, sensitivity analysis.*

---

<a name='graphrag' />

# GraphRAG Q&A Pipeline

<a href=#top>↑ Return to Top of Notebook</a>

*Placeholder — structured retrieval Q&A interface over Neo4j graph.*
